# CARE Patch Generation (Real Data, Nested Layout)

This notebook generates training patches (`.npz`) for CARE / csbdeep training from *real* microscopy datasets stored in a nested folder structure. **Specifically for data from ISS preprocessing pipeline**. 

### What it does
For every detected sample folder, it:
1. Finds matching pairs between a **source** directory (e.g. `raw`) and a **target** directory (e.g. `rlf50` or `rlf25`) by **relative path**.
2. Splits data into two patch datasets:
   - **NON_DAPI**: all channels except the user-specified DAPI channel index
   - **DAPI_ONLY**: only the DAPI channel
3. Excludes files that do **not** contain a `_chN.tif` pattern.
4. Runs `csbdeep.data.create_patches(...)` to create patches and saves `.npz` files.

### Expected folder layout (per sample)
A *sample folder* is any directory that contains both `SOURCE_DIRNAME` and `TARGET_DIRNAME`.

`<CARE_ROOT>/<any_structure>/<sample>/{SOURCE_DIRNAME,TARGET_DIRNAME}/R*/preprocessing/Cycle*/4_retiled/*.tif`

Example:
- `/home/sagah/moldia-archive/CARE_training_data/Victoria/BCR_TCR_CDR3/tonsil/<sample>/raw/R1/preprocessing/Cycle1/4_retiled/Cycle1_s0_ch0.tif`
- `/home/sagah/moldia-archive/CARE_training_data/Victoria/BCR_TCR_CDR3/tonsil/<sample>/rlf50/R1/preprocessing/Cycle1/4_retiled/Cycle1_s0_ch0.tif`

Matching is done by relative path, e.g.:
- `raw/R1/preprocessing/Cycle1/4_retiled/Cycle1_s0_ch0.tif`
- `rlf50/R1/preprocessing/Cycle1/4_retiled/Cycle1_s0_ch0.tif`

### Outputs
Per sample (stored in `<sample>/train_patches/`):
- `<group>__<sample>__NON_DAPI__train_patches.npz`
- `<group>__<sample>__DAPI_ONLY__train_patches.npz`

Optional merged outputs (stored in `<CARE_ROOT>/train_patches/`):
- `ALL_SAMPLES__NON_DAPI__train_patches.npz`
- `ALL_SAMPLES__DAPI_ONLY__train_patches.npz`

### Next step
Use the produced `.npz` file(s) in the CARE training notebook (e.g. train main model on NON_DAPI, and train a separate model on DAPI_ONLY).

## Imports

In [4]:
from pathlib import Path

from ISS_CARE.ISS_CARE_datagen import (
    run_patch_generation,
    visualize_saved_patches_across_samples,
)

## User settings

Edit all user-configurable parameters in the **User settings** cell at the top of the notebook.

These include:

- `CARE_ROOT`: root folder containing all CARE training data  
- `CARE_SUBDIRS`: top-level subdirectories to process  
      - set to `[]` to include **all subdirectories under `CARE_ROOT`**  
      - or specify manually, e.g. `["christina", "Victoria"]`
- `SOURCE_DIRNAME` / `TARGET_DIRNAME`: folder names inside each sample directory  
- `PATTERN`: recursive pattern used to find image tiles  
- `DAPI_CHANNEL_INDEX`: 0-based channel index for DAPI  

### Sampling settings (per sample)
- `MAX_IMAGES_PER_SAMPLE`: fixed number of image pairs to use per sample (e.g. `200–600`)  
- `FRACTION_IMAGES_PER_SAMPLE`: fraction of image pairs to use (e.g. `0.25` for 25%)
   
    - If both are set, `MAX_IMAGES_PER_SAMPLE` takes priority  
    - Set both to `None` to use all images  
    - Tip:  
        - use smaller values (e.g. `100–300`) for quick testing  
        - use larger values (e.g. `500+`) for final training  

### Patch settings

- `PATCH_SIZE`: size of extracted patches (e.g. `128x128`)  

- `N_PATCHES_PER_IMAGE`: number of patches sampled per image pair  
  - typical: `3–8` for most datasets  
  - `1–5` → good for sparse / punctate microscopy (your case)  
  - fewer = faster, less redundancy, often **better generalization**  
  - more = larger dataset, but often adds **redundant or low-information patches**  
  - ⚠️ too high → model overfits to background / easy regions  

- `PATCH_FILTER_THRESHOLD`: threshold for filtering background patches  

  Recommended values:
  - `0.0` → no filtering (keeps many background patches)  
  - `0.01` → light filtering (safe baseline)  
  - `0.02–0.05` → ✅ **recommended for your data** (better signal-to-noise in training)  
  - `≥ 0.1` → ⚠️ often too aggressive (may remove real but dim signal)


> **Note on filtering (important for your pipeline):**  
> This pipeline already performs **image-level filtering before patch extraction**, removing pairs where the data is:
> - empty or nearly empty  
> - constant / near-constant  
> - invalid (NaN / Inf)  
>
> So by the time patches are created:
> - your dataset is already **clean at the image level**  
> - `PATCH_FILTER_THRESHOLD` only controls **local patch content**
>
> In practice:
> - low values (0.01) = safer, more data  
> - moderate values (0.02–0.05) = **better training signal (recommended)**  
> - high values (>0.1) = risk of removing useful biological structure  


> **Practical recommendation for this dataset:**  
> Start with:
> ```python
> N_PATCHES_PER_IMAGE = 5
> PATCH_FILTER_THRESHOLD = 0.05
> ```
>
> This typically:
> - reduces background bias  
> - improves denoising quality  
> - avoids over-representing empty regions  
### Axes settings
- `AXES`, `PATCH_AXES`: axis conventions for input images and patches  

### Output settings
- `PATCH_DIRNAME`: folder where patch files are stored  
- `MERGE_ALL_SAMPLES`: whether to combine all samples into one dataset  
- `MERGED_NON_DAPI_NAME`, `MERGED_DAPI_NAME`: filenames for merged outputs  

In [5]:
# ----------------------------
# Paths and dataset
# ----------------------------

HOME = Path.home()
CARE_ROOT = HOME / "moldia-archive" / "CARE_training"
CARE_SUBDIRS = ["christina", "Victoria"]

SOURCE_DIRNAME = "raw"
TARGET_DIRNAME = "rlf50"
PATTERN = "**/4_retiled/*.tif"
DAPI_CHANNEL_INDEX = 4


# ----------------------------
# Sampling
# ----------------------------

MAX_IMAGES_PER_SAMPLE = 100
FRACTION_IMAGES_PER_SAMPLE = None
SAMPLING_SEED = 42


# ----------------------------
# Image / pair filtering
# ----------------------------

CHECK_HALF_PLANE_ARTIFACTS = True
CHECK_SIGNAL_CONSISTENCY = True
CHECK_LOW_INFORMATION_TARGET = True

MIN_SOURCE_MAX = 0.0
MIN_SOURCE_STD = 1e-6
MIN_TARGET_MAX = 0.0
MIN_TARGET_STD = 1e-6
EXTREME_VALUE_CUTOFF = 1e6

SIGNAL_STD_THRESHOLD = 3e-3
EMPTY_STD_THRESHOLD = 1e-6
SIGNAL_MAX_THRESHOLD = 0.0
EMPTY_MAX_THRESHOLD = 0.0

TARGET_ROBUST_RANGE_FLOOR = 1e-3
MIN_TARGET_TO_SOURCE_ROBUST_RANGE_RATIO = 0.12
MIN_TARGET_TO_SOURCE_STD_RATIO = 0.15


# ----------------------------
# Patch generation
# ----------------------------

AXES = "YX"
PATCH_AXES = "YX"
PATCH_SIZE = (128, 128)
N_PATCHES_PER_IMAGE = 1
PATCH_FILTER_THRESHOLD = 0.12
PATCH_DIRNAME = "train_patches"


# ----------------------------
# Signal-aware pair selection
# ----------------------------

PAIR_SELECTION_MODE = "signal_biased"   # or "uniform"
SIGNAL_SCORE_USE_TARGET = True
MAX_PAIR_REPEATS = 3
DROP_LOW_SCORE_FRACTION = 0.20
MIN_PAIRS_TO_KEEP_AFTER_SIGNAL_BIAS = 8

SIGNAL_SCORE_PMIN = 1.0
SIGNAL_SCORE_PMAX = 99.0
SIGNAL_FOREGROUND_FRACTION = 0.10

SIGNAL_WEIGHT_STD = 1.5
SIGNAL_WEIGHT_ROBUST_RANGE = 1.5
SIGNAL_WEIGHT_FOREGROUND = 3.0
SIGNAL_WEIGHT_MAX = 0.25


# ----------------------------
# Output
# ----------------------------

MERGE_ALL_SAMPLES = True
MERGED_NON_DAPI_NAME = "NON_DAPI_train_patches_Leica_40_signal_biased_stronger.npz"
MERGED_DAPI_NAME = "DAPI_ONLY_train_patches_Leica_40_signal_biased_stronger.npz"

## Run patch generation

Run this cell to generate CARE training patches from your data.

This step will:

- automatically detect all valid **sample directories**
- match `raw` and `target` images by **relative path**
- optionally **subsample images per sample**
- split data into:
  - **NON_DAPI** (main model)
  - **DAPI_ONLY** (separate model)
- generate training patches using `csbdeep`
- save `.npz` patch files per sample
- optionally create **merged datasets across all samples**
- automatically generate and save a **metadata file** with all parameters and outputs

Outputs include:
- per-sample patch files in `<sample>/train_patches/`
- optional merged datasets in `<CARE_ROOT>/train_patches/`
- metadata file:  
  `<CARE_ROOT>/train_patches/patch_generation_metadata.json`

⚠️ Depending on dataset size, this step can take time.

In [ ]:
results = run_patch_generation(
    care_root=CARE_ROOT,
    care_subdirs=CARE_SUBDIRS,
    source_dirname=SOURCE_DIRNAME,
    target_dirname=TARGET_DIRNAME,
    pattern=PATTERN,
    dapi_channel_index=DAPI_CHANNEL_INDEX,
    axes=AXES,
    patch_size=PATCH_SIZE,
    n_patches_per_image=N_PATCHES_PER_IMAGE,
    patch_axes=PATCH_AXES,
    patch_filter_threshold=PATCH_FILTER_THRESHOLD,
    patch_dirname=PATCH_DIRNAME,
    merge_all_samples=MERGE_ALL_SAMPLES,
    merged_non_dapi_name=MERGED_NON_DAPI_NAME,
    merged_dapi_name=MERGED_DAPI_NAME,
    max_images_per_sample=MAX_IMAGES_PER_SAMPLE,
    fraction_images_per_sample=FRACTION_IMAGES_PER_SAMPLE,
    sampling_seed=SAMPLING_SEED,

    # Image / pair filtering
    min_source_max=MIN_SOURCE_MAX,
    min_source_std=MIN_SOURCE_STD,
    min_target_max=MIN_TARGET_MAX,
    min_target_std=MIN_TARGET_STD,
    extreme_value_cutoff=EXTREME_VALUE_CUTOFF,
    check_half_plane_artifacts=CHECK_HALF_PLANE_ARTIFACTS,
    check_signal_consistency=CHECK_SIGNAL_CONSISTENCY,
    check_low_information_target=CHECK_LOW_INFORMATION_TARGET,
    signal_std_threshold=SIGNAL_STD_THRESHOLD,
    empty_std_threshold=EMPTY_STD_THRESHOLD,
    signal_max_threshold=SIGNAL_MAX_THRESHOLD,
    empty_max_threshold=EMPTY_MAX_THRESHOLD,
    target_robust_range_floor=TARGET_ROBUST_RANGE_FLOOR,
    min_target_to_source_robust_range_ratio=MIN_TARGET_TO_SOURCE_ROBUST_RANGE_RATIO,
    min_target_to_source_std_ratio=MIN_TARGET_TO_SOURCE_STD_RATIO,

    # Signal-aware pair selection
    pair_selection_mode=PAIR_SELECTION_MODE,
    signal_score_use_target=SIGNAL_SCORE_USE_TARGET,
    max_pair_repeats=MAX_PAIR_REPEATS,
    drop_low_score_fraction=DROP_LOW_SCORE_FRACTION,
    min_pairs_to_keep_after_signal_bias=MIN_PAIRS_TO_KEEP_AFTER_SIGNAL_BIAS,
    signal_score_pmin=SIGNAL_SCORE_PMIN,
    signal_score_pmax=SIGNAL_SCORE_PMAX,
    signal_foreground_fraction=SIGNAL_FOREGROUND_FRACTION,
    signal_weight_std=SIGNAL_WEIGHT_STD,
    signal_weight_robust_range=SIGNAL_WEIGHT_ROBUST_RANGE,
    signal_weight_foreground=SIGNAL_WEIGHT_FOREGROUND,
    signal_weight_max=SIGNAL_WEIGHT_MAX,
)

Starting CARE patch generation
Samples detected: 4

[Sample 1/4] 20250116_tonsil_S14_VDJ_new


## Visualize generated patches

Run this cell to inspect a few example patch pairs from each sample.

This will:

- load the saved `.npz` patch files
- randomly select a few patch pairs per sample
- display **input (top)** and **target (bottom)** images
- apply normalization for better visibility

This is useful to:
- verify that source and target are correctly aligned  
- check patch quality and signal content  
- spot potential issues before training  


In [ ]:
# Visualize all samples
visualize_saved_patches_across_samples(
    results["all_patch_files_non_dapi"],
    n_show_per_sample=10,
    variant_name="NON_DAPI",
    random_seed=42,
)